# Efficient Data Handling through Data Parallelism

Unexecuted notebook. Run on remote Jupyter/Colab with Java 17, Spark 3.5.7 and requirements-remote.txt installed. See docs/EXECUTION.md. Upload source and real bank.csv. Save the executed notebook with outputs.

1. **Load the**: 01_preview.csv and printed rows demonstrate validated Spark DataFrame loading.
2. **Implement a method**: Round-robin repartition(n) balances row counts without concentrating common job categories on one partition. 01_partition_sizes_N.csv records actual partition sizes. Aggregation later shuffles by job; adaptive execution can coalesce shuffle partitions.
3. **Identify and calculate**: 02_parallel_job_balance.csv uses distributed groupBy(job).avg(balance); benchmark repeats partition grids without replicating the dataset. Compare actual medians and invariant aggregate outputs.
4. **Perform a parallel operation**: Loan amount is absent: loan is yes/no. 02_top5_loan_holder_age_bands.csv ranks decade age bands by personal-loan-holder counts and shows average account balance separately. This is a labeled proxy, not a solution for unavailable loan amounts.
5. **Choose a classification**: Logistic regression uses distributed optimization and offers interpretable coefficients. It handles the binary subscription target; the main modeling module supplies cross-validation.
6. **Partition the dataset**: 03_04_partitioned_model.json records the partitioned training model and held-out metrics. Approximately 80/20 deterministic split; training repartitions to the largest grid size. Explain shuffle overhead, skew, sparse feature size, and driver/executor memory limits.
7. **Implement resource monitoring**: resource_samples.json samples driver Python CPU/RSS, local JVM child RSS, host CPU/memory each second. Spark UI/event logs supply executor CPU time, GC, memory/spill, shuffle, and task metrics. Local driver metrics do not measure remote executors; fill observations from actual runs.
8. **Manage multiple parallel tasks**: 06_scheduled_tasks.json comes from two bounded independent preprocessing actions launched with InheritableThread and separate FAIR scheduler pools/job groups. CV parallelism is independently limited to avoid oversubscription. Show Spark UI jobs/stages and discuss measured resource contention.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd()
if not (REPO / 'src').exists():
    REPO = REPO.parent
assert (REPO / 'src').exists(), 'Set REPO to your uploaded repository root'
sys.path.insert(0, str(REPO / 'src'))
INPUT = REPO / 'data' / 'bank.csv'
OUTPUT = REPO / 'artifacts' / 'notebook-parallelism-01'
EVENT_DIR = None  # optional existing event-log URI on your compute host
assert INPUT.is_file()
assert not OUTPUT.exists(), 'Choose a new output path'
OUTPUT.mkdir(parents=True)
from spark_common import session, load_bank, Monitor, save_json
spark = session('notebook-parallelism', partitions=2, event_dir=EVENT_DIR, master='local[2]')
spark.sparkContext.addPyFile(str(REPO / 'src' / 'banklib.py'))
save_json(OUTPUT / 'run_config.json', {'input': str(INPUT), 'spark_version': spark.version,
          'master': spark.sparkContext.master, 'seed_split': 2026, 'fold_hash_seed': 91})
print(spark.version)


In [ ]:
data = load_bank(spark, str(INPUT))
data.printSchema()
data.show(10)
data.groupBy('y').count().show()


In [ ]:
from parallelism import run
with Monitor(OUTPUT / 'resource_samples.json'):
    run(data, OUTPUT, partitions=(1, 2, 4), repeats=3)


In [ ]:
for path in sorted(OUTPUT.glob('*.json')):
    print(path.name)
    print(path.read_text()[:12000])
from IPython.display import display, Image
for path in sorted(OUTPUT.glob('*.png')):
    display(Image(filename=str(path)))


## Observations to complete after execution
Record actual values, explain patterns and limitations, and reference saved artifacts.
Parallelism: include medians, CPU/RSS peaks, executor metrics and overhead.
Modeling: compare PR-AUC/recall and majority baseline, discuss imbalance and leakage.


In [ ]:
spark.stop()
